<a href="https://colab.research.google.com/github/chewanna7-code/NatureInsightStudy/blob/main/Subcatchment_Analysis_Merged.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Subcatchment Intervention Summary and Hydrograph Plotter

This notebook supports the subcatchment analysis section of the River Wansbeck NatureInsight workflow.

The subcatchments were manually divided in SCALGO using the **Subwatershed** tool and an Environment Agency overlay. The purpose of this notebook is to summarise how NatureInsight intervention opportunities vary between the upland, midland and lowland sections of the catchment.

The notebook also includes an optional hydrograph plotting workflow for comparing NatureInsight scenario hydrographs exported as Excel files.

## What this notebook produces

1. A table of intervention counts by catchment zone.
2. A total intervention count chart for each subcatchment zone.
3. A proportional stacked bar chart showing intervention composition.
4. Optional pie charts for visual comparison.
5. Optional hydrograph overlay plot from uploaded NatureInsight scenario files.

## Notes

The intervention counts are entered manually from the SCALGO/NatureInsight outputs. If new outputs are exported, only the data table needs updating.




In [ ]:
# Import required libraries

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from openpyxl import load_workbook
import io
import re
import warnings

warnings.filterwarnings("ignore")

# Set a simple academic-style font
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 9

print("Libraries loaded.")

## 1. Define Subcatchment Intervention Data

The table below records the number of NatureInsight intervention opportunities identified in each catchment zone.

The zones represent the manually defined subcatchments used in the analysis:

- **Upland**
- **Midland**
- **Lowland**
- **Overall Catchment**

Update the values in this cell if the NatureInsight outputs are re-exported or if the subcatchment boundaries are adjusted.

In [ ]:
# Define intervention counts by catchment zone

data = {
    "Intervention": [
        "Runoff Attenuation Feature",
        "Floodplain Reconnection",
        "Large Woody Debris",
        "Tree Planting",
        "Wet Woodland",
        "Buffer Strip",
        "Soil Management",
        "Gully Stuffing",
        "Grip Blocking"
    ],

    # Counts from the SCALGO/NatureInsight subcatchment outputs
    "Upland":  [815, 226, 452, 250, 186, 463, 185, 198, 93],
    "Midland": [362, 65, 75, 50, 4, 565, 523, 21, 7],
    "Lowland": [26, 13, 66, 162, 0, 100, 413, 11, 0],

    # Overall catchment total
    "Overall Catchment": [1202, 304, 591, 452, 190, 1126, 1121, 242, 100]
}

df = pd.DataFrame(data).set_index("Intervention")

# Display the input table
display(df)

## 2. Set Intervention Colours

These colours are used consistently across the graphs so that each intervention type is easier to track between figures.

In [ ]:
# Define colours for each intervention type

colors = {
    "Runoff Attenuation Feature": "#0072B2",
    "Floodplain Reconnection": "#56B4E9",
    "Large Woody Debris": "#8B0000",
    "Tree Planting": "#228B22",
    "Wet Woodland": "#005F5B",
    "Buffer Strip": "#2ECC71",
    "Soil Management": "#BDB76B",
    "Gully Stuffing": "#8B5A2B",
    "Grip Blocking": "#F8766D"
}

print("Colours assigned.")

## 3. Total Interventions by Subcatchment Zone

This figure compares the total number of NatureInsight intervention opportunities identified in each subcatchment zone.

The overall catchment total is excluded from this figure so that the upland, midland and lowland zones can be compared directly.

In [ ]:
# Plot total intervention count by catchment zone

zone_totals = df[["Upland", "Midland", "Lowland"]].sum()

plt.figure(figsize=(8, 5))

bars = plt.bar(
    zone_totals.index,
    zone_totals.values,
    width=0.5,
    color="#4C72B0"
)

plt.title("Total Nature-Based Solution Interventions by Catchment Zone", pad=14)
plt.ylabel("Total number of interventions")
plt.xlabel("Catchment zone")

# Add values above bars
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 40,
        f"{int(height):,}",
        ha="center",
        va="bottom",
        fontsize=11
    )

plt.tight_layout()
plt.savefig("subcatchment_total_interventions.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: subcatchment_total_interventions.png")

## 4. Proportional Intervention Composition

This figure shows the proportional composition of intervention types within each zone.

This is useful because the subcatchment zones are not equal in size or opportunity count. Using percentages allows the intervention mix to be compared more fairly than using raw totals alone.

In [ ]:
# Calculate proportional composition by zone

df_prop = df.div(df.sum(axis=0), axis=1) * 100

fig, ax = plt.subplots(figsize=(11, 6))

bottom = np.zeros(len(df_prop.columns))

for intervention in df_prop.index:
    values = df_prop.loc[intervention]

    ax.bar(
        df_prop.columns,
        values,
        bottom=bottom,
        label=intervention,
        color=colors[intervention],
        width=0.6
    )

    bottom += values.values

plt.title("Proportional Intervention Composition by Catchment Zone", pad=14)
plt.ylabel("Percentage of interventions (%)")
plt.xlabel("Catchment zone")
plt.xticks(rotation=0)

plt.legend(
    title="Intervention type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.savefig("subcatchment_intervention_composition.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: subcatchment_intervention_composition.png")

## 5. Optional Pie Chart Comparison

The pie charts provide an additional visual comparison of intervention composition by zone.

These are optional because the stacked bar chart is usually clearer for reporting, but the pie charts can be useful for quick visual summaries.

In [ ]:
# Optional pie chart grid for each catchment zone

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
zones = ["Upland", "Midland", "Lowland", "Overall Catchment"]

for ax, zone in zip(axes.flatten(), zones):
    values = df[zone]

    # Remove zero values for cleaner pie charts
    values = values[values > 0]

    ax.pie(
        values,
        labels=None,
        startangle=90,
        colors=[colors[i] for i in values.index]
    )

    ax.set_title(zone, fontsize=14)

fig.legend(
    df.index,
    loc="center right",
    title="Intervention type"
)

plt.suptitle(
    "Proportional Distribution of Nature-Based Solutions Across Catchment Zones",
    fontsize=16
)

plt.tight_layout(rect=[0, 0, 0.85, 0.95])
plt.savefig("subcatchment_pie_charts.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: subcatchment_pie_charts.png")

## 6. Optional Hydrograph Overlay Plotter

This optional section allows NatureInsight hydrograph Excel files to be uploaded and plotted together.

It is useful where multiple subcatchment or intervention scenario hydrographs need to be compared on the same time axis.

The code searches each uploaded Excel file for total time and total flow columns, extracts the hydrograph, and plots all loaded scenarios together.

In [ ]:
# Optional hydrograph upload and extraction functions

try:
    from google.colab import files
    COLAB_AVAILABLE = True
except ImportError:
    COLAB_AVAILABLE = False

def extract_hydrograph(fbytes):
    """Extract total time and total flow data from a NatureInsight hydrograph Excel file."""

    wb = load_workbook(io.BytesIO(fbytes), data_only=True)
    ws = wb.active

    total_time_col = None
    total_flow_col = None
    header_row = None

    # Search worksheet for the total time and total flow columns
    for row in ws.iter_rows():
        for cell in row:
            val = str(cell.value).strip().lower() if cell.value else ""

            if "total" in val and "time" in val and "sec" in val:
                total_time_col = cell.column
                header_row = cell.row

            if "total" in val and "flow" in val and "time" not in val:
                total_flow_col = cell.column

        if total_time_col and total_flow_col:
            break

    if not total_time_col or not total_flow_col:
        return None, None, None, None

    times = []
    flows = []

    # Read values below the header row
    for ri in range(header_row + 1, ws.max_row + 1):
        t = ws.cell(ri, total_time_col).value
        f = ws.cell(ri, total_flow_col).value

        if t is not None and f is not None:
            try:
                times.append(float(t) / 3600)
                flows.append(float(f))
            except ValueError:
                pass

    if not flows:
        return None, None, None, None

    peak = max(flows)
    time_to_peak = times[flows.index(peak)]

    return times, flows, peak, time_to_peak

def parse_label(fname):
    """Create a simple plot label from the uploaded filename."""

    name = fname.replace(".xlsx", "").replace(".XLSX", "")
    rp_match = re.search(r"(\d+)Y", name.upper())
    rp = rp_match.group(1) + "Y" if rp_match else ""

    for tag in ["OBS", "OBH", "OBC", "ALL", "BASE", "BASELINE", "RUN", "FP", "SM", "LU", "RAF", "FPR"]:
        if tag in name.upper():
            return f"{tag} {rp}" if rp else tag

    return name[:30]

print("Hydrograph functions ready.")

## 7. Upload Hydrograph Files

Run this cell if hydrograph comparison is required. Upload all NatureInsight hydrograph Excel files at once when prompted.

If you only need the subcatchment intervention summary, this section can be skipped.

In [ ]:
# Upload hydrograph Excel files

if COLAB_AVAILABLE:
    print("Upload all hydrograph Excel files now...")
    uploaded = files.upload()

    hydrographs = {}

    for fname, fbytes in uploaded.items():
        times, flows, peak, time_to_peak = extract_hydrograph(fbytes)

        if times:
            label = parse_label(fname)
            hydrographs[label] = (times, flows, peak, time_to_peak)
            print(f"{label}: peak = {peak:.2f} m³/s at {time_to_peak:.2f} hours")
        else:
            print(f"Warning: could not read {fname}")

    print(f"\nLoaded {len(hydrographs)} hydrographs.")
else:
    print("File upload is only available in Google Colab.")

## 8. Configure and Plot Hydrographs

Edit the plot settings if needed, then run the cell to generate the hydrograph overlay figure.

In [ ]:
# Configure and plot hydrograph overlay

PLOT_TITLE = "NatureInsight Modelled Hydrographs with NbS Intervention"
PLOT_SUBTITLE = "Wansbeck Catchment"
Y_LABEL = "Flow (m³/s)"
X_LABEL = "Time (hours)"
X_LIMIT_HRS = 55

COLOUR_MAP = {
    "BLUE": "#2980b9",
    "GREEN": "#27ae60",
    "DARKERGREEN": "#1a5c38",
    "CYAN": "#17a589",
    "ORANGE": "#e67e22",
    "RED": "#c0392b",
    "PURPLE": "#8e44ad",
    "PINK": "#e91e8c",
    "BROWN": "#784212",
    "GREY": "#7f8c8d",
    "GRAY": "#7f8c8d",
    "YELLOW": "#f1c40f",
    "NAVY": "#1a1a2e",
    "BLACK": "#2c3e50"
}

auto_colours = list(plt.cm.tab10.colors)

if "hydrographs" in globals() and len(hydrographs) > 0:
    fig, ax = plt.subplots(figsize=(15, 7))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("#f9f9f9")

    styles = ["-", "--", "-.", ":"]

    for i, (label, (times, flows, peak, time_to_peak)) in enumerate(sorted(hydrographs.items())):
        fname_upper = label.upper()

        col = next(
            (hex_col for colour_name, hex_col in COLOUR_MAP.items() if colour_name in fname_upper),
            auto_colours[i % len(auto_colours)]
        )

        sty = styles[i // len(auto_colours) % len(styles)]

        ax.plot(
            times,
            flows,
            sty,
            color=col,
            linewidth=1.8,
            alpha=0.88,
            label=f"{label} (peak = {peak:.1f} m³/s)"
        )

        ax.scatter(
            [time_to_peak],
            [peak],
            color=col,
            s=70,
            zorder=5,
            edgecolors="white",
            linewidth=1.2
        )

    ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.7)
    ax.set_axisbelow(True)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    ax.set_xlabel(X_LABEL, fontsize=12, labelpad=8)
    ax.set_ylabel(Y_LABEL, fontsize=12, labelpad=8)

    if X_LIMIT_HRS:
        ax.set_xlim(0, X_LIMIT_HRS)

    ax.set_ylim(bottom=0)

    ax.legend(
        fontsize=8,
        framealpha=0.9,
        edgecolor="#cccccc",
        loc="upper center",
        ncol=1,
        bbox_to_anchor=(0.8, 1.2)
    )

    fig.suptitle(PLOT_TITLE, fontsize=14, fontweight="bold", x=0.07, ha="left", y=1.02)
    fig.text(0.07, 0.97, PLOT_SUBTITLE, fontsize=10, color="#555555", style="italic")
    fig.text(
        0.07,
        0.01,
        "NI outputs: NatureInsight® (ARUP|SCALGO, 2024). Analysis: Author.",
        fontsize=8.5,
        color="#888888"
    )

    plt.subplots_adjust(top=0.88, bottom=0.10, left=0.07, right=0.97)
    plt.savefig("hydrograph_overlay.png", dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()

    print("Saved: hydrograph_overlay.png")
else:
    print("No hydrographs loaded. Upload files in the previous cell if this output is needed.")

## 9. Download Outputs

In Google Colab, run the cell below to download the generated figures.

In [ ]:
# Download generated outputs in Google Colab

if COLAB_AVAILABLE:
    output_files = [
        "subcatchment_total_interventions.png",
        "subcatchment_intervention_composition.png",
        "subcatchment_pie_charts.png",
        "hydrograph_overlay.png"
    ]

    for output_file in output_files:
        try:
            files.download(output_file)
            print(f"Downloaded: {output_file}")
        except FileNotFoundError:
            print(f"Skipped: {output_file} was not created.")
else:
    print("Download helper is only available in Google Colab.")